In [38]:
import pandas as pd
import numpy as np
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import RobustScaler
from math import log as ln

In [39]:
sland = pd.read_parquet('data/parquets/scotland.parquet')
gland = pd.read_parquet('data/parquets/greenland.parquet')

datasets = [gland, sland]

full_dataset = pd.concat(datasets, ignore_index=True)

In [40]:
sland

,geometry,x,y,elevation,roughness_15,roughness_150,roughness_1500,roughness_3000,temperature_tasJan,temperature_tasFeb,...,precipitation_prJul,precipitation_prAug,precipitation_prSep,precipitation_prOct,precipitation_prNov,precipitation_prDec,precipitation_total,bedrock_bedrock,landscape classification_classifica,time since deglaciation_time since
307,b'\x01\x01\x00\x00\x00\xc9x\xc6\x03\\\xa0\x05A...,177163.501843,868987.192828,132.580002,3.061677,1.588694,2.161997,1.312703,4.700,4.975,...,102.200001,131.875000,151.574997,181.400002,195.424999,196,1749.974997,sedimentary,areal scour,28
956,b'\x01\x01\x00\x00\x00\x18\xfan\x02\xfa\x17\x0...,140031.251188,834368.651955,242.539993,1.243724,3.729259,2.517404,1.569288,3.375,4.275,...,169.875000,203.450001,210.375000,293.500000,289.199997,305,2712.024992,igneous,mountain valley,28
781,b'\x01\x01\x00\x00\x00J\xfc\xb3\xe7A\x1c\x13A\...,313104.476273,637293.718031,436.079987,0.349012,3.175493,8.035336,4.183985,2.075,2.850,...,86.375000,96.074999,79.299999,110.575001,117.950001,126,1134.124998,sedimentary,unmodified,28
610,b'\x01\x01\x00\x00\x00\xa0\xbb\x8aA9\xf9\x08A\...,204583.157003,863252.935608,296.700012,1.228277,2.207848,6.665524,4.717516,2.750,2.675,...,124.324997,169.200001,195.450001,245.000000,247.149998,288,2437.924997,sedimentary,mountain valley,15
70,b'\x01\x01\x00\x00\x00\xe4\xb4$9\xfd\xe5\x08A\...,203967.652902,561766.016980,54.959999,0.639478,2.821668,1.647959,0.884659,4.700,5.000,...,85.300003,94.099998,94.400002,139.699997,136.500000,135,1205.625010,sedimentary,None,28


In [ ]:
def prep_data(df):
    dropped = df.drop(['x', 'y', 'elevation', 'roughness_15', 'roughness_150', 'roughness_1500', 'roughness_3000'], axis=1)
    return dropped

In [ ]:
glandX = prep_data(gland)
slandX = prep_data(sland)
fullX = prep_data(full_dataset)

In [ ]:
categoricals = [ 'bedrock_metamorphic',
                 'bedrock_sedimentary',
                 'classification_linear erosion',
                 'classification_mountain valley',
                 'classification_unmodified']

continuous = ['precipitation'
              ,'temperature'
                ]

def scale_data(df):
    cats = df[categoricals]
    conts = df[continuous]
    scaler = RobustScaler()
    conts_scaled = scaler.fit_transform(conts)
    scaled_df = pd.DataFrame(conts_scaled, index=conts.index, columns=conts.columns)
    final_df = pd.concat([scaled_df, cats], axis=1)
    return final_df

In [ ]:
gxscaled = scale_data(glandX)
sxscaled = scale_data(slandX)
fullxscaled = scale_data(fullXX)

gY15 = gland['roughness_15']
gY150 = gland['roughness_150']
gY1500 = gland['roughness_1500']
gY3000 = gland['roughness_3000']
sY15 = sland['roughness_15']
sY150 = sland['roughness_150']
sY1500 = sland['roughness_1500']
sY3000 = sland['roughness_3000']
fY15 = full_dataset['roughness_15']
fY150 = full_dataset['roughness_150']
fY1500 = full_dataset['roughness_1500']
fY3000 = full_dataset['roughness_3000']


In [ ]:
def get_tsd_dict(data):
    tsd = list(data['time since deglaciation'].unique())
    tsd.sort()

    tsd_dict = {}

    for i in tsd:
        tsd_dict[i] = data[data['time since deglaciation'] == i]
    return tsd_dict

def prep_tsd(data):
    prepped_tsd = {}
    for key in data.keys():
        prepped_tsd[key] = prep_data(data[key])
    return prepped_tsd

def scale_tsd(data):
    tsd_scaled = {} 
    for key in data.keys():
        tsd_scaled[str(key)] = scale_data(data[key])
    return tsd_scaled

def get_x(data):
    tsd_dict = get_tsd_dict(data)
    prepped_tsd = prep_tsd(tsd_dict)
    scaled_tsd = scale_tsd(prepped_tsd)
    return scaled_tsd

def get_vif(df):
    vif = pd.DataFrame()
    vif['Feature'] = df.columns
    vif['VIF'] = [variance_inflation_factor(df, i) for i in range(df.shape[1])]
    return vif

resolutions = ['15', '150', '1500', '3000']

def get_ys(data):
    ys = {}
    tsd_dict = get_tsd_dict(data)
    for res in resolutions:
        for key in tsd_dict.keys():
            ys[str(key) + 'y' + res] = tsd_dict[key]['roughness_' + res]
    return ys

def run_ols(df, dfy):
    df_constant = sm.add_constant(df)
    df_model = sm.OLS(dfy, df_constant).fit()
    return df_model

def partial_r_squared(df, dfy):
    values = []
    for item in list(df.columns):
        newdf = df.copy()
        if 'bedrock' in item:
            newdf.drop(['bedrock_metamorphic', 'bedrock_sedimentary'], axis=1, inplace=True)
        elif 'classification' in item:
            newdf.drop(['classification_linear erosion', 'classification_mountain valley', 'classification_unmodified'], axis=1, inplace=True)
        else:
            newdf.drop([item], axis=1, inplace=True)
        #newdf.drop([item], axis=1, inplace=True)
        ols = run_ols(newdf, dfy)
        rsquare = ols.rsquared_adj
        values.append(rsquare)
    return values

def get_model_summary(df, dfy, vif):
    model_results = run_ols(df, dfy)
    overall_r = model_results.rsquared_adj
    df_summary = model_results.params.drop('const')
    partial_r = partial_r_squared(df, dfy)
    new_vif = vif.copy()
    new_vif['OLS Correlation Coefficient'] = df_summary.to_frame().reset_index(drop=True)
    new_vif['Partial R'] = partial_r
    new_vif['Full R'] = overall_r
    new_vif['Partial R Squared'] = ((new_vif['Full R'] - new_vif['Partial R']) / (1 - new_vif['Partial R'])).round(4)
    # new_vif.drop(['VIF'], axis=1, inplace=True)
    return new_vif

# Workaround to avoid fixing the 27.0 dataset issue...
def get_model_summary_27(df, dfy, vif):
    model_results = run_ols(df, dfy)
    overall_r = model_results.rsquared_adj
    df_summary = model_results.params
    partial_r = partial_r_squared(df, dfy)
    new_vif = vif.copy()
    new_vif['OLS Correlation Coefficient'] = df_summary.to_frame().reset_index(drop=True)
    new_vif['Partial R'] = partial_r
    new_vif['Full R'] = overall_r
    new_vif['Partial R Squared'] = ((new_vif['Full R'] - new_vif['Partial R']) / (1 - new_vif['Partial R'])).round(4)
    # new_vif.drop(['VIF'], axis=1, inplace=True)
    return new_vif


# There's something odd going on here in that the 27.0 dataset doesn't generate a 'const' value. Not sure why. Have excluded it for now, but would be good to resolve.
def get_summaries(x_dataset, y_dataset, vif, resolutions=resolutions):
    summary_dict = {}
    for resolution in resolutions:
        if resolution == '15':
            for y in y_dataset.keys():
                if 'y150' not in y and 'y3000' not in y:
                    for x in x_dataset.keys():
                        if x == y[:-3] and x != '27.0':
                            summary_dict[x + '_' + resolution] = get_model_summary(x_dataset[x], y_dataset[y], vif)
                        elif x == y[:-3] and x == '27.0':
                            summary_dict[x + '_' + resolution] = get_model_summary_27(x_dataset[x], y_dataset[y], vif)
        elif resolution == '150':
            for y in y_dataset.keys():
                if 'y150' in y and 'y1500' not in y and 'y3000' not in y:
                    for x in x_dataset.keys():
                        if x == y[:-4] and x != '27.0':
                            summary_dict['+' + x + '_' + resolution] = get_model_summary(x_dataset[x], y_dataset[y], vif)
                        elif x == y[:-4] and x == '27.0':
                            summary_dict[x + '_' + resolution] = get_model_summary_27(x_dataset[x], y_dataset[y], vif)
        elif resolution == '1500':
            for y in y_dataset.keys():
                if 'y1500' in y:
                    for x in x_dataset.keys():
                        if x == y[:-5] and x != '27.0':
                            summary_dict['*' + x + '_' + resolution] = get_model_summary(x_dataset[x], y_dataset[y], vif)
                        elif x == y[:-5] and x == '27.0':
                            summary_dict[x + '_' + resolution] = get_model_summary_27(x_dataset[x], y_dataset[y], vif)
        else:
            for y in y_dataset.keys():
                if 'y15' not in y:
                    for x in x_dataset.keys():
                        if x == y[:-5] and x != '27.0':
                            summary_dict['_' + x + '_' + resolution] = get_model_summary(x_dataset[x], y_dataset[y], vif)
                        elif x == y[:-5] and x == '27.0':
                            summary_dict[x + '_' + resolution] = get_model_summary_27(x_dataset[x], y_dataset[y], vif)
    return summary_dict

def mega_summary(data):
    x = get_x(data)
    tsd = list(data['time since deglaciation'].unique())
    tsd.sort()
    vif = get_vif(x[str(tsd[0])])
    y = get_ys(data)
    summaries = get_summaries(x, y, vif=vif)
    return summaries

def extract_r2(dataset, filename):
    # Create a new dataframe to hold the summary data
    data = pd.DataFrame()

    # Add columns for time since deglaciation and resolution
    data['time'] = []
    data['resolution'] = []
    single_key = list(dataset.keys())[0]

    # Add columns for each variable
    for v in range(len(dataset[single_key]['Feature'])):
        data[dataset[single_key]['Feature'][v]] = []

    # Iterate through each list in the dictionary and pull out the values of R2, then add them to the dataframe
    for i in range(len(dataset)):
        # Extract the dictionary key
        key = list(dataset.keys())[i]
        # Use the key to pull out the R2 values for each variable
        r2_data = [dataset[key]['Partial R Squared'].iloc[v] for v in range(len(dataset[key]['Feature']))]
        # Get rid of characters at start of key
        key_stripped = key.lstrip('+*_')
        # Add row to dataframe including appropriately formatted values for time and resolution columns
        data.loc[len(data)] = [float(key_stripped.split('_')[0]), float(key_stripped.split('_')[1])] + r2_data

    # Save data to csv
    #data.to_csv(f'{filename}.csv')
    
    return data

In [ ]:
gland_summary = mega_summary(gland)
sland_summary = mega_summary(sland)
massive_summary = mega_summary(massive)

In [ ]:
gr2 = extract_r2(gland_summary, 'gland')
sr2 = extract_r2(sland_summary, 'sland')
mr2 = extract_r2(massive_summary, 'massive')

# dfs = [gr2, sr2]
# merged = pd.concat(dfs, ignore_index=True)

In [ ]:
def f(row):
    if row['resolution'] == 15.0:
        val = 'red' 
    elif row['resolution'] == 150.0:
        val = 'yellow'
    elif row['resolution'] == 1500.0:
        val = 'green'
    else:
        val = 'blue'
    return val

def add_colours(dataset):
    dataset['colour'] = dataset.apply(f, axis=1)
    return dataset

In [ ]:
add_colours(gr2)
add_colours(sr2)
add_colours(mr2)

In [ ]:
def get_best_fit(x, y):
    coefficients = np.polyfit(x, y, 1)
    slope, intercept = coefficients
    best_fit_line = slope * x + intercept
    return best_fit_line

In [ ]:
gr2_15, gr2_150, gr2_1500, gr2_3000 = gr2[gr2['resolution'] == 15.0], gr2[gr2['resolution'] == 150.0], gr2[gr2['resolution'] == 1500.0], gr2[gr2['resolution'] == 3000.0]
sr2_15, sr2_150, sr2_1500, sr2_3000 = sr2[sr2['resolution'] == 15.0], sr2[sr2['resolution'] == 150.0], sr2[sr2['resolution'] == 1500.0], sr2[sr2['resolution'] == 3000.0]
mr2_15, mr2_150, mr2_1500, mr2_3000 = mr2[mr2['resolution'] == 15.0], mr2[mr2['resolution'] == 150.0], mr2[mr2['resolution'] == 1500.0], mr2[mr2['resolution'] == 3000.0]

In [ ]:
mr2['landscape classification'] = mr2['classification_linear erosion']

In [ ]:
rows = mr2[mr2['time'] > 18].index
mr2pruned = mr2.drop(rows)
mr2pruned

In [ ]:
mr2pruned_15, mr2pruned_150, mr2pruned_1500, mr2pruned_3000 = mr2pruned[mr2pruned['resolution'] == 15.0], mr2pruned[mr2pruned['resolution'] == 150.0].reset_index(drop=True), mr2pruned[mr2pruned['resolution'] == 1500.0].reset_index(drop=True), mr2pruned[mr2pruned['resolution'] == 3000.0].reset_index(drop=True)

In [ ]:
pruned_dataframes = [mr2pruned_15, mr2pruned_150, mr2pruned_1500, mr2pruned_3000]

In [ ]:
def get_rate_of_decay(dataset):
    a = dataset['landscape classification'][0]
    y = dataset['landscape classification'][16]
    t = 18 - 6.5
    rate_of_decay = (ln(y/a))/t
    return rate_of_decay


get_rate_of_decay(mr2pruned_3000)


In [ ]:
decay_rates = pd.DataFrame({'Resolution': [15, 150, 1500, 3000], 'Rate of decay': [round(get_rate_of_decay(df), 2) for df in pruned_dataframes]})

decay_rates

In [ ]:
fig, axs = plt.subplots(1, 2, layout='constrained', figsize=(16, 8), sharey=True
)

plt.rcParams.update({'font.size': 20})

ax1, ax2 = axs[0], axs[1]

ax1.annotate('a', xy=(0.95, .95), xycoords='axes fraction', fontsize=20, fontweight='bold')
ax2.annotate('b', xy=(0.95, .95), xycoords='axes fraction', fontsize=20, fontweight='bold')

x = 'time'
y = 'precipitation'
z = 'landscape classification'
#z = 'classification_mountain valley'
colours = ['red', 'yellow', 'green', 'blue']


def plotit(axa, axb, dataseta, datasetb):
    axa.scatter(dataseta[x], dataseta[z], c=dataseta['colour'], edgecolor='k', lw=0.5, zorder=5)
    axb.scatter(datasetb[x], datasetb[y], c=datasetb['colour'], edgecolor='k', lw=0.5, zorder=5)
    axa.grid(alpha=0.5, zorder=-1)
    axb.grid(alpha=0.5, zorder=-1)

plotit(ax1, ax2, mr2pruned, mr2pruned)

tsd = list(mr2['resolution'].unique())
tsd.sort()

patches = []

for i in range(len(tsd)):
    patches.append(mpatches.Patch(color=colours[i], label=tsd[i]))

ax2.legend(handles=patches, title='Scale (m)', fontsize='small')

ax1.set_title('Landscape Classification')
ax2.set_title('Total Annual Precipitation (mm)')

plt.rc('xtick', labelsize=20) 
plt.rc('ytick', labelsize=20) 

fig.supylabel('Partial $R^2$', fontsize=25)
fig.supxlabel('Time since deglaciation (thousand years)', fontsize=25)

plt.savefig('/home/jamiemac/Documents/University/Loughborough/PhD/Project 1/Figures/precipitation_vs_classification_4.png', dpi=300)

In [ ]:
svif = get_vif(sxscaled)
gvif = get_vif(gxscaled)
mvif = get_vif(mxscaled)

In [ ]:
summarym15 = get_model_summary(mxscaled, mY15, mvif)
summarym150 = get_model_summary(mxscaled, mY150, mvif)
summarym1500 = get_model_summary(mxscaled, mY1500, mvif)
summarym3000 = get_model_summary(mxscaled, mY3000, mvif)

In [ ]:
summaryg15 = get_model_summary(gxscaled, gY15, gvif)
summaryg150 = get_model_summary(gxscaled, gY150, gvif)
summaryg1500 = get_model_summary(gxscaled, gY1500, gvif)
summaryg3000 = get_model_summary(gxscaled, gY3000, gvif)
summarys15 = get_model_summary(sxscaled, sY15, svif)
summarys150 = get_model_summary(sxscaled, sY150, svif)
summarys1500 = get_model_summary(sxscaled, sY1500, svif)
summarys3000 = get_model_summary(sxscaled, sY3000, svif)

In [ ]:

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(20, 20), sharex=True, sharey=True, layout='constrained')
plt.rcParams.update({'font.size': 20})

ax1 = axes[0, 0]
ax1.annotate('15 m', xy=(0.82, .95), xycoords='axes fraction', fontsize=20, fontweight='bold')
ax2 = axes[0, 1]
ax2.annotate('150 m', xy=(0.82, .95), xycoords='axes fraction', fontsize=20, fontweight='bold')
ax3 = axes[1, 0]
ax3.annotate('1500 m', xy=(0.82, .95), xycoords='axes fraction', fontsize=20, fontweight='bold')
ax4 = axes[1, 1]
ax4.annotate('3000 m', xy=(0.82, .95), xycoords='axes fraction', fontsize=20, fontweight='bold')

ax1.scatter('OLS Correlation Coefficient', 'Feature', data=summaryg15, s=100, c='blue', alpha=0.7, edgecolors='k')
ax1.scatter('OLS Correlation Coefficient', 'Feature', data=summarys15, marker='X', s=100, c='red', alpha=0.7, edgecolors='k')
# ax1.scatter('OLS Correlation Coefficient', 'Feature', data=summarygn15, s=100, c='lime', marker='X', alpha=0.7, edgecolors='k')
# ax1.scatter('OLS Correlation Coefficient', 'Feature', data=summarygs15, s=100, c='fuchsia', marker='D', alpha=0.7, edgecolors='k')

ax2.scatter('OLS Correlation Coefficient', 'Feature', data=summaryg150, s=100, c='blue', alpha=0.7, edgecolors='k')
ax2.scatter('OLS Correlation Coefficient', 'Feature', data=summarys150, marker='X', s=100, c='red', alpha=0.7, edgecolors='k')
# ax2.scatter('OLS Correlation Coefficient', 'Feature', data=summarygn150, s=100, c='lime', marker='X', alpha=0.7, edgecolors='k')
# ax2.scatter('OLS Correlation Coefficient', 'Feature', data=summarygs150, s=100, c='fuchsia', marker='D', alpha=0.7, edgecolors='k')

ax3.scatter('OLS Correlation Coefficient', 'Feature', data=summaryg1500, s=100, c='blue', alpha=0.7, edgecolors='k')
ax3.scatter('OLS Correlation Coefficient', 'Feature', data=summarys1500, marker='X', s=100, c='red', alpha=0.7, edgecolors='k')
# ax3.scatter('OLS Correlation Coefficient', 'Feature', data=summarygn1500, s=100, c='lime', marker='X', alpha=0.7, edgecolors='k')
# ax3.scatter('OLS Correlation Coefficient', 'Feature', data=summarygs1500, s=100, c='fuchsia', marker='D', alpha=0.7, edgecolors='k')

ax4.scatter('OLS Correlation Coefficient', 'Feature', data=summaryg3000, s=100, c='blue', alpha=0.7, edgecolors='k')
ax4.scatter('OLS Correlation Coefficient', 'Feature', data=summarys3000, marker='X', s=100, c='red', alpha=0.7, edgecolors='k')
# ax4.scatter('OLS Correlation Coefficient', 'Feature', data=summarygn3000, s=100, c='lime', marker='X', alpha=0.7, edgecolors='k')
# ax4.scatter('OLS Correlation Coefficient', 'Feature', data=summarygs3000, s=100, c='fuchsia', marker='D', alpha=0.7, edgecolors='k')

ax1.grid(alpha=0.5, zorder=-1)
ax1.axvline(0, color='black', lw=1, ls='--', zorder=-6)

ax2.grid(alpha=0.5, zorder=-1)
ax2.axvline(0, color='black', lw=1, ls='--', zorder=-6)

ax3.grid(alpha=0.5, zorder=-1)
ax3.axvline(0, color='black', lw=1, ls='--', zorder=-6)

ax4.grid(alpha=0.5, zorder=-1)
ax4.axvline(0, color='black', lw=1, ls='--', zorder=-6)

ax2.legend(['Greenland', 'Scotland'
            #, 'North Greenland', 'South Greenland'
            ], loc='center right')

plt.rc('xtick', labelsize=20) 
plt.rc('ytick', labelsize=20) 

fig.supylabel('Variable', fontsize=25)
fig.supxlabel('OLS Correlation Coefficient', fontsize=25)

In [ ]:
def getr2forclassification(scale):
    totalr2 = scale['Partial R Squared'][4] + scale['Partial R Squared'][5] + scale['Partial R Squared'][6]
    return totalr2.round(3)

getr2forclassification(summarys3000)

In [ ]:
greenland = [summaryg15, summaryg150, summaryg1500, summaryg3000]
scotland = [summarys15, summarys150, summarys1500, summarys3000]
precipitation = [0]
temperature = [1]
climate = [0, 1]
bedrock = [2, 3]
classification = [4, 5, 6]

def get_avg(area, variables):
    R2 = 'Partial R Squared'
    mean = 0
    for variable in variables:
        mean += (area[0][R2][variable] + area[1][R2][variable] + area[2][R2][variable] + area[3][R2][variable]) / 4
    return mean.round(3)

mean_values = pd.DataFrame({'Region': ['Greenland', 'Scotland'], 
'Precipitation': [get_avg(greenland, precipitation), get_avg(scotland, precipitation)],
'Temperature': [get_avg(greenland, temperature), get_avg(scotland, temperature)],
'Climate': [get_avg(greenland, climate), get_avg(scotland, climate)],
'Bedrock': [get_avg(greenland, bedrock), get_avg(scotland, bedrock)],
'Classification': [get_avg(greenland, classification), get_avg(scotland, classification)]})

mean_values

In [ ]:
summaries = [summarys15, summaryg15, summarys150, summaryg150, summarys1500, summaryg1500, summarys3000, summaryg3000]

for summary in summaries:
    summary.loc[len(summary)] = ['landscape\nclassification', 'Nan', 'Nan', 'Nan', 'Nan', summary['Partial R Squared'][4]]
    summary.loc[len(summary)] = ['bedrock', 'Nan', 'Nan', 'Nan', 'Nan', summary['Partial R Squared'][2]]
    summary.drop([2, 3, 4, 5, 6], inplace=True)

summarys15

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(20, 20), sharex=True, 
                         #sharey=True, 
                         layout='constrained')
plt.rcParams.update({'font.size': 20})

ax1 = axes[0, 0]
ax1.annotate('15 m', xy=(0.82, .95), xycoords='axes fraction', fontsize=20, fontweight='bold')
ax2 = axes[0, 1]
ax2.annotate('150 m', xy=(0.82, .95), xycoords='axes fraction', fontsize=20, fontweight='bold')
ax3 = axes[1, 0]
ax3.annotate('1500 m', xy=(0.82, .95), xycoords='axes fraction', fontsize=20, fontweight='bold')
ax4 = axes[1, 1]
ax4.annotate('3000 m', xy=(0.82, .95), xycoords='axes fraction', fontsize=20, fontweight='bold')

variables = (summaryg15['Feature'])
r2values = {
    'Greenland': (summaryg15['Partial R Squared']),
    'Scotland': (summarys15['Partial R Squared'])
}

x = np.arange(len(variables))  # the label locations
width = 0.25  # the width of the bars
offset = 0.25

def plotsy(ax, df1, df2):
    ax.bar(x, df1['Partial R Squared'], width, zorder=5, edgecolor='k')
    ax.bar(x + offset, df2['Partial R Squared'], width, zorder=5, edgecolor='k')
    ax.set_xticks(x + width, variables)

plotsy(ax1, summaryg15, summarys15)
plotsy(ax2, summaryg150, summarys150)
plotsy(ax3, summaryg1500, summarys1500)
plotsy(ax4, summaryg3000, summarys3000)

axes = [ax1, ax2, ax3, ax4]

for ax in axes:
    ax.grid(alpha=0.5, zorder=-1)
    ax.tick_params('x', rotation=90)

ax2.legend(['Greenland', 'Scotland'
            #, 'North Greenland', 'South Greenland'
            ]#, loc='center right'
            )

plt.rc('xtick', labelsize=20) 
plt.rc('ytick', labelsize=20) 

fig.supxlabel('Variable', fontsize=25)
fig.supylabel('Partial $R^2$', fontsize=25)

plt.savefig('/home/jamiemac/Documents/University/Loughborough/PhD/Project 1/Figures/Partial_R2.png', dpi=300)

In [ ]:
test = massive.groupby(by='time since deglaciation').sum()
test.drop(['x', 'y', 'elevation', 'roughness_15', 'roughness_150', 'roughness_1500', 'roughness_3000', 'precipitation', 'temperature', 'bedrock_metamorphic', 'bedrock_sedimentary'], axis=1, inplace=True)
test['total'] = count['x']
test.columns = ['Linear Erosion', 'Mountain Valley', 'Unmodified', 'Total']
test['Areal Scour'] = test['Total'] - (test['Linear Erosion'] + test['Mountain Valley'] + test['Unmodified'])
plotty = test.drop('Total', axis=1)
plotty['Time since deglaciation (kya)'] = plotty.index
fig = plotty.plot(x='Time since deglaciation (kya)', kind='bar', stacked=True, figsize=(17, 17), zorder=5, edgecolor='k')

fig.grid(alpha=0.5, zorder=-1)

plt.rc('xtick', labelsize=20) 
plt.rc('ytick', labelsize=20) 

fig.set_xlabel('Time since deglaciation (thousand years)', fontsize=25)
fig.set_ylabel('Number of points', fontsize=25)

plt.savefig('/home/jamiemac/Documents/University/Loughborough/PhD/Project 1/Figures/tsd_vs_classification.png', dpi=300)

In [ ]:
# import required libraries
import geopandas as gpd
import rioxarray as rio
from pathlib import Path

# # create output directories
# temp_folder = Path('data/rasters/greenland/climate/temperature')
# temp_folder.mkdir(parents=True, exist_ok=True)
# prcp_folder = Path('data/rasters/greenland/climate/precipitation')
# prcp_folder.mkdir(parents=True, exist_ok=True)

# # import climate datasets and national boundaries
# gland = gpd.read_file('data/vectors/greenland/GRL_adm0.shp').to_crs(epsg=4326)
# temp = rio.open_rasterio('data/rasters/greenland/climate/cru_ts4.09.1901.2024.tmp.dat.nc')
# prcp = rio.open_rasterio('data/rasters/greenland/climate/cru_ts4.09.1901.2024.pre.dat.nc')

sland = gpd.read_file('data/vectors/scotland/scotland.shp').to_crs(epsg=27700)
s_temp = gpd.read_file('data/vectors/scotland/Monthly_Temperature_Observations_1991-2020.shp').to_crs(epsg=27700)
s_prcp = gpd.read_file('data/vectors/scotland/Monthly_Precipitation_Observations_1991-2020.shp').to_crs(epsg=27700)

# # assign the greenland shapefile CRS to climate data
# temp.rio.write_crs(gland.crs, inplace=True)
# prcp.rio.write_crs(gland.crs, inplace=True)

# # clip climate data to Greenland boundary
# tclipped = temp.rio.clip(gland.geometry.values, drop=True)
# tclean = tclipped.where(tclipped != tclipped.tmp.rio.nodata)

# pclipped = prcp.rio.clip(gland.geometry.values, drop=True)
# pclean = pclipped.where(pclipped != pclipped.pre.rio.nodata)

# # select data for the period 1991-2020
# t91_20 = tclean.sel(time=slice('1991-01-01', '2020-12-31'))
# p91_20 = pclean.sel(time=slice('1991-01-01', '2020-12-31'))

# # resample to annual data
# annualtmp = t91_20.resample(time='1YE').mean()
# annualprcp = p91_20.resample(time='1YE').sum()

# # calculate mean annual temperature and total annual precipitation
# gtemp = annualtmp['tmp'].mean(dim='time')
# gprcp = annualprcp['pre'].mean(dim='time')

# # reproject to EPSG:3413
# gtemp = gtemp.rio.reproject("EPSG:3413")
# gprcp = gprcp.rio.reproject("EPSG:3413")
# gprcp = gprcp.where(gprcp > 0)  # remove zero values for precipitation

# # write greenland data
# gtemp.rio.to_raster('data/rasters/greenland/climate/temperature/temperature.tif')
# gprcp.rio.to_raster('data/rasters/greenland/climate/precipitation/precipitation.tif')

# clip the UK datasets to the Scotland boundary
stemp = s_temp.clip(sland)
sprcp = s_prcp.clip(sland)

# calculate mean temperature and total precipitation for each grid cell
stemp_values = stemp.select_dtypes(include=['float64', 'int64']).columns
scot_temp = gpd.GeoDataFrame({'mean': stemp[stemp_values].mean(axis=1), 'geometry': stemp['geometry']})
sprcp_values = sprcp.select_dtypes(include=['float64', 'int64']).columns
scot_prcp = gpd.GeoDataFrame({'total': sprcp[sprcp_values].sum(axis=1), 'geometry': sprcp['geometry']})

# # write scottish data to file
# sprcp.to_file('data/vectors/scotland/precipitation.shp')
# stemp.to_file('data/vectors/scotland/temperature.shp')

In [11]:
scot_temp = gpd.GeoDataFrame({'mean': stemp[stemp_values].mean(axis=1), 'geometry': stemp['geometry']})
scot_temp

,mean,geometry
6786,9.741667,"POLYGON ((210000.012 535000.003, 210000.012 53..."
45723,9.758333,"POLYGON ((202000.011 553000.004, 202000.011 55..."
35421,9.800000,"POLYGON ((192000.011 625000.003, 192000.011 62..."
48414,9.764583,"POLYGON ((192000.011 625000.003, 194000.011 62..."
10321,9.166667,"POLYGON ((190000.011 627000.004, 192000.011 62..."
...,...,...
62762,8.587500,"POLYGON ((342000.012 1049696.981, 342470 10496..."
12240,8.591667,"POLYGON ((344000.012 1050200.348, 344300 10502..."
50294,8.591667,"POLYGON ((345945.123 1053000.004, 345900 10526..."
50221,8.572222,"POLYGON ((350000.012 1055000.004, 350000.012 1..."


In [10]:
newdf = gpd.GeoDataFrame({'mean': stemp['mean'], 'geometry': stemp['geometry']})
newdf

,mean,geometry
6786,9.741667,"POLYGON ((210000.012 535000.003, 210000.012 53..."
45723,9.758333,"POLYGON ((202000.011 553000.004, 202000.011 55..."
35421,9.800000,"POLYGON ((192000.011 625000.003, 192000.011 62..."
48414,9.764583,"POLYGON ((192000.011 625000.003, 194000.011 62..."
10321,9.166667,"POLYGON ((190000.011 627000.004, 192000.011 62..."
...,...,...
62762,8.587500,"POLYGON ((342000.012 1049696.981, 342470 10496..."
12240,8.591667,"POLYGON ((344000.012 1050200.348, 344300 10502..."
50294,8.591667,"POLYGON ((345945.123 1053000.004, 345900 10526..."
50221,8.572222,"POLYGON ((350000.012 1055000.004, 350000.012 1..."
